## Predicting the World Cup Winner

In [1]:
# Get libraries
import pandas as pd
import numpy as np
import random 

In [4]:
# Load the soccer data
url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
df = pd.read_csv(url)

# Show the data
print(df.head())
print(len(df))

         date home_team away_team  home_score  away_score tournament     city  \
0  1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1  1873-03-08   England  Scotland           4           2   Friendly   London   
2  1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3  1875-03-06   England  Scotland           2           2   Friendly   London   
4  1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
49520


In [6]:
# Build the ELO ratings functions
teams = {}  # empty teams dictionary

def expected_score(r1, r2):
    return 1 / (1 + 10**((r2-r1)/400))

def update_elo(r1, r2, score1, score2, k=30):
    e1 = expected_score(r1, r2)

    # Determine actual match results
    if score1 > score2:
        s1 = 1  # Team 1 wins
    elif score1 < score2:
        s1 = 0  # Team 1 loses
    else:
        s1 = 0.5

    # Apply the ELO update formula
    new_r1 = r1 + k * (s1 - e1)
    new_r2 = r2 + k * ((1 - s1) - (1-e1))

    return new_r1, new_r2

In [7]:
# Process every match in the dataset
for _, row in df.iterrows():
    t1, t2 = row['home_team'], row['away_team']  # team names
    s1, s2 = row['home_score'], row['away_score'] # final scores

    # If the team is NOT in the dictionary add it with a default rating of 1000 else do nothing
    teams.setdefault(t1, 1000)
    teams.setdefault(t2, 1000)

    # Update both teams ELO ratings
    teams[t1], teams[t2] = update_elo(teams[t1], teams[t2], s1, s2)

In [8]:
# Convert a dictionary into a list of (team, rating)
team_list = list(teams.items())

# Sort by the ratings (highest first)
team_list.sort(key=lambda item: item[1], reverse=True)

# Take the top 32 teams
top_teams = team_list[:32]

# Extract only the team names
top_teams = [team for team, rating in top_teams]

In [10]:
top_teams

['Spain',
 'Argentina',
 'France',
 'England',
 'Portugal',
 'Brazil',
 'Colombia',
 'Netherlands',
 'Germany',
 'Morocco',
 'Mexico',
 'Japan',
 'Belgium',
 'Italy',
 'Croatia',
 'Norway',
 'Switzerland',
 'Ecuador',
 'Denmark',
 'Turkey',
 'Austria',
 'Uruguay',
 'United States',
 'Iran',
 'Algeria',
 'South Korea',
 'Australia',
 'Canada',
 'Paraguay',
 'Senegal',
 'Russia',
 'Basque Country']

In [9]:
# Simulate a single match
# We will use a poisson model to generate realistic soccer scores
def simulate_match(team1, team2):
    r1, r2 = teams[team1], teams[team2]   # Get the ELO ratings
    prob1 = expected_score(r1, r2)

    # Generate realistic soccer scores
    score1 = np.random.poisson(1.5 * prob1 + 0.3)
    score2 = np.random.poisson(1.5 * (1-prob1) + 0.3)

    # Decide the winner
    if score1 > score2:
        return team1
    elif score2 > score1:
        return team2
    else:
        return random.choice([team1, team2])

In [13]:
# Simulate an entire world cup tournament
def simulate_tournament():
    current = top_teams.copy()  # Getting the top 32 teams
    random.shuffle(current)  # randomize the matchups

    while len(current) > 1:
        next_round = []  # An empty list to contain the teams each round while there is more than one team in the current list
        for i in range(0, len(current), 2):
            winner = simulate_match(current[i], current[i+1])
            next_round.append(winner)
        current = next_round # Advancing only the winners of each match
    return current[0]  # return the winner/champion

In [14]:
# Run the 10,000 simulations
results = {} # track the wins
N = 10000

for _ in range(N):
    champ = simulate_tournament()
    results[champ] = results.get(champ, 0) + 1

In [15]:
# Display the results
predictions = sorted(results.items(), key=lambda x: x[1], reverse=True)

In [18]:
print('World Cup Prediction Results')
print('----------------------------')
for team, wins in predictions[:10]:
    print(f"{team}: {wins/N:.2%} chance to win.")

print('The predicted champion ', predictions[0][0])

World Cup Prediction Results
----------------------------
Spain: 15.32% chance to win.
Argentina: 12.82% chance to win.
France: 7.40% chance to win.
England: 6.38% chance to win.
Portugal: 4.86% chance to win.
Brazil: 4.65% chance to win.
Colombia: 4.06% chance to win.
Germany: 3.81% chance to win.
Netherlands: 3.73% chance to win.
Morocco: 3.57% chance to win.
The predicted champion  Spain
